### Problem Statement 3B

3B. Implement Industrial Equipment Anomaly Detection on the MVTec AD Kaggle dataset using Autoencoder and build the model by using the following steps:
Steps:
- Preprocess images (resize, normalize) for input into a convolutional autoencoder.
- Build a convolutional autoencoder (CNN-based encoder-decoder) to learn the representations of normal object images.
- Use reconstruction errors in pixel values to detect anomalies in test images.
Dataset: MVTec AD is an anomaly detection dataset for industrial defect detection, containing images of normal and defective objects.
Goal: Detect defects in objects by training an image-based autoencoder on images of defect-free objects.

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose

In [ ]:
# Preprocess images (resize, normalize) for input into a convolutional autoencoder
# Load normal images from workspace
folder_path = "../LP IV Updated Datasets/Object Detection/fashion mnist/Ankle-boot"

image_files = [folder_path + "/" + f for f in os.listdir(folder_path) if f.endswith('.png')][:200]
print(f"Loading {len(image_files)} sample normal object images...")

IMG_SIZE = 32
normal_images = []

for p in image_files:
    img = Image.open(p).convert('L').resize((IMG_SIZE, IMG_SIZE))
    normal_images.append(np.array(img) / 255.0)

normal_images = np.array(normal_images).reshape(-1, IMG_SIZE, IMG_SIZE, 1)

# Create simulated defective images by adding black defect spots
defective_images = normal_images.copy()[:50]
for img in defective_images:
    img[12:18, 12:18, 0] = 0.0  # Defect patch

# Split normal images into train and test
X_train = normal_images[:150]
X_test_normal = normal_images[150:]

# Test set contains both normal and defective images
X_test = np.vstack([X_test_normal, defective_images])
y_test = np.array([0] * len(X_test_normal) + [1] * len(defective_images))

print("X_train normal images shape:", X_train.shape)
print("X_test total images shape:", X_test.shape)

In [ ]:
# Build a convolutional autoencoder (CNN-based encoder-decoder)
model = Sequential()

# Encoder
model.add(Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)))
model.add(MaxPooling2D((2, 2), padding='same'))

# Decoder
model.add(Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same'))
model.add(Conv2D(1, (3, 3), activation='sigmoid', padding='same'))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train the convolutional autoencoder on normal (defect-free) objects
model.fit(X_train, X_train, epochs=5, batch_size=16, validation_split=0.1)

In [ ]:
# Use reconstruction errors in pixel values to detect anomalies in test images
reconstructions = model.predict(X_test)

# Calculate Mean Squared Error per image
mse = np.mean(np.square(X_test - reconstructions), axis=(1, 2, 3))

# Set threshold at 95th percentile of normal test images
threshold = np.percentile(mse[y_test == 0], 95)
print("Anomaly detection threshold:", threshold)

# Predict anomalies (error > threshold is an anomaly)
y_pred = (mse > threshold).astype(int)
print(f"Detected {sum(y_pred)} anomalies out of {len(y_test)} test images.")

# Display sample normal and defective reconstruction comparison
plt.figure(figsize=(8, 4))

plt.subplot(2, 2, 1)
plt.imshow(X_test[0].squeeze(), cmap='gray')
plt.title("Normal Original")

plt.subplot(2, 2, 2)
plt.imshow(reconstructions[0].squeeze(), cmap='gray')
plt.title("Normal Reconstructed")

plt.subplot(2, 2, 3)
plt.imshow(X_test[-1].squeeze(), cmap='gray')
plt.title("Defective Original")

plt.subplot(2, 2, 4)
plt.imshow(reconstructions[-1].squeeze(), cmap='gray')
plt.title("Defective Reconstructed")

plt.tight_layout()
plt.show()